In [3]:
import pandas as pd
from scipy import stats

file_path = "Копия Данные для тестового задания.xlsx"

audience = pd.read_excel(file_path, sheet_name="Данные об аудитории")
ab_tests = pd.read_excel(file_path, sheet_name="Данные АБ тестов")
listers = pd.read_excel(file_path, sheet_name="Листеры")

print("Размер таблицы с аудиторией:", audience.shape)
print("Размер таблицы A/B-тестов:", ab_tests.shape)
print("Размер таблицы с листерами:", listers.shape)


Размер таблицы с аудиторией: (16814, 3)
Размер таблицы A/B-тестов: (2835, 4)
Размер таблицы с листерами: (158, 6)


## Первичный просмотр данных

In [4]:
display(audience.head())
display(ab_tests.head())
display(listers.head())

print("\nПропуски в данных об аудитории:")
display(audience.isna().sum())


,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


,experiment_num,experiment_group,user_id,revenue
0,1,test,38456,520
1,1,control,13125924,806
2,1,control,9761984,0
3,1,test,11387012,208
4,1,test,18319648,104


,user_id,date,cnt_adverts,age,cnt_contacts,revenue
0,100,2022-01-01,6,21,119,53
1,100,2022-01-02,2,21,200,18
2,100,2022-01-03,6,21,193,42
3,100,2022-01-04,2,21,143,38
4,100,2022-01-05,2,21,190,40



Пропуски в данных об аудитории:


date            0
user_id         0
view_adverts    0
dtype: int64

## Задание 1. MAU

MAU — количество уникальных пользователей, которые были активны в течение месяца.


In [5]:
mau = audience["user_id"].nunique()
print(f"MAU = {mau:,}".replace(",", " "))


MAU = 7 639


**Ответ:** MAU = **7 639 пользователей**.


## Задание 2. DAU

Сначала считаю число уникальных пользователей для каждого дня, затем беру среднее за ноябрь.


In [6]:
audience["date"] = pd.to_datetime(audience["date"])

dau_by_day = (
    audience.groupby("date")["user_id"]
    .nunique()
)

average_dau = dau_by_day.mean()

print(dau_by_day.head())
print(f"\nСредний DAU = {average_dau:.2f}")
print(f"Округленный DAU = {round(average_dau)}")


date
2023-11-01    623
2023-11-02    649
2023-11-03    573
2023-11-04    343
2023-11-05    350
Name: user_id, dtype: int64

Средний DAU = 560.47
Округленный DAU = 560


**Ответ:** средний DAU ≈ **560 пользователей в день**.


## Задание 3. Retention первого дня для пользователей, пришедших 1 ноября

1. Определяю первую дату активности каждого пользователя.
2. Выбираю пользователей, впервые пришедших 1 ноября.
3. Проверяю, сколько из них вернулось 2 ноября.


In [7]:
first_visit = audience.groupby("user_id")["date"].min()

nov_1 = pd.Timestamp("2023-11-01")
nov_2 = pd.Timestamp("2023-11-02")

new_users_nov_1 = set(first_visit[first_visit == nov_1].index)
users_nov_2 = set(audience.loc[audience["date"] == nov_2, "user_id"])

returned_users = new_users_nov_1 & users_nov_2

retention_d1 = len(returned_users) / len(new_users_nov_1) * 100

print("Новых пользователей 1 ноября:", len(new_users_nov_1))
print("Вернулись 2 ноября:", len(returned_users))
print(f"Retention D1 = {retention_d1:.2f}%")


Новых пользователей 1 ноября: 623
Вернулись 2 ноября: 166
Retention D1 = 26.65%


**Ответ:** Retention D1 = **26,6%**.


## Задание 5. Пользовательская конверсия в просмотр объявления

Целевое действие — пользователь посмотрел хотя бы одно объявление (`view_adverts > 0`).

Так как один пользователь может встречаться в данных несколько раз, сначала суммирую просмотры по пользователю.


In [8]:
views_by_user = audience.groupby("user_id")["view_adverts"].sum()

users_with_views = (views_by_user > 0).sum()
total_users = views_by_user.shape[0]

conversion = users_with_views / total_users * 100

print("Всего пользователей:", total_users)
print("Пользователей с просмотром объявления:", users_with_views)
print(f"Конверсия = {conversion:.2f}%")


Всего пользователей: 7639
Пользователей с просмотром объявления: 3538
Конверсия = 46.31%


**Ответ:** пользовательская конверсия = **46,3%**.


## Задание 6. Среднее количество просмотренных объявлений на пользователя

Использую уже рассчитанное количество просмотров для каждого пользователя и беру среднее.


In [9]:
avg_views = views_by_user.mean()

print(f"Среднее количество просмотров на пользователя = {avg_views:.2f}")


Среднее количество просмотров на пользователя = 2.87


**Ответ:** в среднем пользователь просмотрел **2,9 объявления**.


## Задание 7. NPS

По условию:
- 2 000 пользователей;
- 1 200 сторонников;
- 500 критиков;
- 300 нейтралов.

Формула:  
**NPS = % сторонников − % критиков**


In [10]:
total = 2000
promoters = 1200
detractors = 500

nps = promoters / total * 100 - detractors / total * 100
print(f"NPS = {nps:.0f}%")


NPS = 35%


**Ответ:** NPS = **35%**.

## Задание 8. A/B-тесты по ARPU

ARPU — средняя выручка на одного пользователя.

Для каждого эксперимента сравниваю `revenue` тестовой и контрольной групп.  
Использую **t-test Уэлча**, потому что сравниваются средние двух независимых групп, а равенство дисперсий заранее не предполагаю.

Уровень значимости: **α = 0.05**.


In [11]:
results = []

for experiment in sorted(ab_tests["experiment_num"].dropna().unique()):
    exp_data = ab_tests[ab_tests["experiment_num"] == experiment]

    control = exp_data.loc[
        exp_data["experiment_group"] == "control", "revenue"
    ]
    test = exp_data.loc[
        exp_data["experiment_group"] == "test", "revenue"
    ]

    t_stat, p_value = stats.ttest_ind(
        test,
        control,
        equal_var=False
    )

    results.append({
        "experiment": int(experiment),
        "control_arpu": control.mean(),
        "test_arpu": test.mean(),
        "difference_%": (test.mean() / control.mean() - 1) * 100,
        "p_value": p_value
    })

ab_results = pd.DataFrame(results)
ab_results.round({
    "control_arpu": 2,
    "test_arpu": 2,
    "difference_%": 2,
    "p_value": 4
})


,experiment,control_arpu,test_arpu,difference_%,p_value
0,1,722.46,665.74,-7.85,0.6890
1,2,704.65,332.93,-52.75,0.0011
2,3,663.21,998.67,50.58,0.0603


### Выводы по A/B-тестам

**Эксперимент 1:**  
ARPU контрольной группы ≈ **722,46**, тестовой ≈ **665,74**.  
p-value ≈ **0,689 > 0,05**. Статистически значимых различий нет. На основании этого теста изменение внедрять не стоит.

**Эксперимент 2:**  
ARPU контрольной группы ≈ **704,65**, тестовой ≈ **332,93**.  
p-value ≈ **0,0011 < 0,05**. Различие статистически значимо, при этом тестовая группа показывает заметно меньший ARPU. **Тестовый вариант не рекомендую внедрять.**

**Эксперимент 3:**  
ARPU контрольной группы ≈ **663,21**, тестовой ≈ **998,67**.  
p-value ≈ **0,0603 > 0,05**. В тестовой группе ARPU выше, но на уровне значимости 5% результат пока нельзя считать статистически значимым. **Можно продолжить эксперимент или увеличить выборку.**


## Задание 9. Средний доход на пользователя по датасету с листерами

Один пользователь встречается в таблице несколько раз, поэтому сначала суммирую его `revenue`, а затем считаю среднее по пользователям.


In [12]:
revenue_by_user = listers.groupby("user_id")["revenue"].sum()
avg_revenue_per_user = revenue_by_user.mean()

print("Количество уникальных пользователей:", revenue_by_user.shape[0])
print(f"Средний доход на пользователя = {avg_revenue_per_user:.2f}")


Количество уникальных пользователей: 31
Средний доход на пользователя = 156.48


**Ответ:** средний доход на пользователя ≈ **156,4**.


## Задание 10. Медиана возраста пользователя

Поскольку пользователь может встречаться в таблице несколько раз, сначала оставляю по одному значению возраста для каждого пользователя.


In [13]:
age_by_user = listers.groupby("user_id")["age"].first()
median_age = age_by_user.median()

print(f"Медиана возраста = {median_age:.0f}")


Медиана возраста = 28


**Ответ:** медиана возраста = **28 лет**.

## Задание 18. Сравнение конверсии в платеж

Дано:
- вариант A: 100 047 501 посетитель и 1 003 платежа;
- вариант B: 100 001 055 посетителей и 1 099 платежей.

Для сравнения двух конверсий использую **z-test для двух пропорций**. Расчёт выполняю через формулу, чтобы не подключать дополнительную библиотеку.


In [14]:
import math

visitors_a = 100_047_501
payments_a = 1_003

visitors_b = 100_001_055
payments_b = 1_099

conversion_a = payments_a / visitors_a
conversion_b = payments_b / visitors_b

# pooled proportion
pooled_p = (payments_a + payments_b) / (visitors_a + visitors_b)

standard_error = math.sqrt(
    pooled_p * (1 - pooled_p) * (1 / visitors_a + 1 / visitors_b)
)

z_stat = (conversion_b - conversion_a) / standard_error
p_value = 2 * stats.norm.sf(abs(z_stat))

uplift = (conversion_b / conversion_a - 1) * 100

print(f"Конверсия A = {conversion_a * 100:.6f}%")
print(f"Конверсия B = {conversion_b * 100:.6f}%")
print(f"Относительный uplift B = {uplift:.2f}%")
print(f"z-statistic = {z_stat:.3f}")
print(f"p-value = {p_value:.4f}")


Конверсия A = 0.001003%
Конверсия B = 0.001099%
Относительный uplift B = 9.62%
z-statistic = 2.105
p-value = 0.0353


### Вывод

Вариант B показывает конверсию выше примерно на **9,6% относительно варианта A**.  
Полученное **p-value ≈ 0,035 < 0,05**, поэтому различие статистически значимо на уровне 5%.

**Рекомендация:** вариант B можно рассматривать для внедрения, дополнительно оценив экономическую значимость эффекта и убедившись, что дизайн эксперимента был корректным.
